# Accuracy Under Different Grading Schemes (no training)

This notebook recomputes accuracy under three grading schemes from the predictions already saved by the fold experiments. No training is required. The purpose is a fair comparison with prior work, which reported a 4-category result on a single dataset, and to quantify how much of the gap to higher accuracy comes from the difficult KL-1 grade versus from the multi-dataset setting.

Three schemes are reported per held-out dataset:
1. Five-class (KL 0–4), as originally measured.
2. Four-category with KL-3 and KL-4 merged into one "severe" class (the prior-work scheme).
3. Four-category with KL-0 and KL-1 merged (removing the ambiguous "doubtful" boundary).

## Setup

Reads the saved prediction files. Each contains the true grades, predicted grades, and 5-class probability vectors for one held-out dataset.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import numpy as np, glob, os, json
from pathlib import Path
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score

MT_RES = Path('/content/drive/MyDrive/Master Thesis/scope3_mt/results')
RES_DIR = Path('/content/drive/MyDrive/Master Thesis/scope3/results')
files = sorted(glob.glob(str(MT_RES/'*_preds.npz')) + glob.glob(str(RES_DIR/'fold*_preds.npz')))
print('Prediction files:')
for f in files: print('  ', os.path.basename(f))

Mounted at /content/drive
Prediction files:
   fold1_test_mendeley_seed0_preds.npz
   fold1_test_mendeley_seed1_preds.npz
   fold1_test_mendeley_seed2_preds.npz
   fold2_test_mrkr_seed0_preds.npz
   fold2_test_mrkr_seed1_preds.npz
   fold2_test_mrkr_seed2_preds.npz
   fold3_test_nhanes3_seed0_preds.npz
   fold3_test_nhanes3_seed1_preds.npz
   fold3_test_nhanes3_seed2_preds.npz
   fold4_test_oai_seed0_preds.npz
   fold4_test_oai_seed1_preds.npz
   fold4_test_oai_seed2_preds.npz
   mt_mendeley_seed0_preds.npz
   mt_mrkr_seed0_preds.npz
   mt_nhanes3_seed0_preds.npz
   mt_nhanes3_v2_seed0_preds.npz
   mt_oai_seed0_preds.npz
   mtc_oai_seed0_preds.npz


## Re-grading

For each saved file, the per-image predicted probabilities are re-binned under each scheme and accuracy, weighted F1, and quadratic weighted kappa are recomputed. Merging classes is applied identically to the true labels and the predictions, so the comparison is consistent.

In [ ]:
def merge(y, scheme):
    y = np.asarray(y).copy()
    if scheme == '5class':
        return y
    if scheme == 'merge34':          # KL3+KL4 -> 3  (prior-work scheme)
        y[y == 4] = 3
        return y
    if scheme == 'merge01':          # KL0+KL1 -> 0, then shift
        y[y == 1] = 0
        y[y >= 2] = y[y >= 2] - 1     # relabel to 0..3
        return y

def scheme_metrics(yt, yp, scheme):
    a = merge(yt, scheme); b = merge(yp, scheme)
    return (accuracy_score(a, b),
            f1_score(a, b, average='weighted'),
            cohen_kappa_score(a, b, weights='quadratic'))

rows = {}
for f in files:
    name = os.path.basename(f).replace('_preds.npz','')
    d = np.load(f)
    yt, yp = d['y_true'], d['y_pred']
    rows[name] = {s: scheme_metrics(yt, yp, s) for s in ['5class','merge34','merge01']}
    print(f'{name}: {len(yt):,} images')
print('Done.')

fold1_test_mendeley_seed0: 8,259 images
fold1_test_mendeley_seed1: 8,259 images
fold1_test_mendeley_seed2: 8,259 images
fold2_test_mrkr_seed0: 39,967 images
fold2_test_mrkr_seed1: 39,967 images
fold2_test_mrkr_seed2: 39,967 images
fold3_test_nhanes3_seed0: 4,785 images
fold3_test_nhanes3_seed1: 4,785 images
fold3_test_nhanes3_seed2: 4,785 images
fold4_test_oai_seed0: 8,547 images
fold4_test_oai_seed1: 8,547 images
fold4_test_oai_seed2: 8,547 images
mt_mendeley_seed0: 8,259 images
mt_mrkr_seed0: 39,967 images
mt_nhanes3_seed0: 4,785 images
mt_nhanes3_v2_seed0: 4,785 images
mt_oai_seed0: 8,547 images
mtc_oai_seed0: 8,547 images
Done.


## Results — accuracy under each scheme

For every held-out dataset, accuracy is shown for the original five-class scheme, the prior-work four-category scheme (KL-3/4 merged), and the four-category scheme that removes the ambiguous KL-1 boundary (KL-0/1 merged). The mean row is the headline.

In [ ]:
def short(n):
    return n.replace('mtc_','fused_').replace('mt_','').replace('_seed0','').replace('fold1_test_','').replace('fold2_test_','').replace('fold3_test_','').replace('fold4_test_','')

print('='*78)
print('ACCURACY BY GRADING SCHEME (external / held-out data)')
print('='*78)
print(f"{'Dataset':18s}{'5-class':>12s}{'4-cat (KL3+4)':>16s}{'4-cat (KL0+1)':>16s}")
print('-'*78)
agg = {'5class':[], 'merge34':[], 'merge01':[]}
for name, r in rows.items():
    print(f"{short(name):18s}{r['5class'][0]:>12.3f}{r['merge34'][0]:>16.3f}{r['merge01'][0]:>16.3f}")
    for s in agg: agg[s].append(r[s][0])
print('-'*78)
print(f"{'MEAN':18s}{np.mean(agg['5class']):>12.3f}{np.mean(agg['merge34']):>16.3f}{np.mean(agg['merge01']):>16.3f}")
print('='*78)
print()
print('4-cat (KL3+4) = prior-work scheme (severe grades merged).')
print('4-cat (KL0+1) = ambiguous "doubtful" grade merged into normal.')
print()
print('QWK (5-class -> 4-cat KL0+1), mean:',
      f"{np.mean([rows[n]['5class'][2] for n in rows]):.3f} -> {np.mean([rows[n]['merge01'][2] for n in rows]):.3f}")

json.dump({n:{s:list(v) for s,v in r.items()} for n,r in rows.items()},
          open(str(MT_RES/'scheme_comparison.json'),'w'), indent=2)
print('Saved -> scheme_comparison.json')

ACCURACY BY GRADING SCHEME (external / held-out data)
Dataset                5-class   4-cat (KL3+4)   4-cat (KL0+1)
------------------------------------------------------------------------------
mendeley                 0.338           0.411           0.419
mendeley_seed1           0.359           0.423           0.449
mendeley_seed2           0.382           0.435           0.475
mrkr                     0.476           0.528           0.523
mrkr_seed1               0.487           0.543           0.527
mrkr_seed2               0.482           0.538           0.523
nhanes3                  0.608           0.626           0.687
nhanes3_seed1            0.612           0.631           0.689
nhanes3_seed2            0.598           0.616           0.674
oai                      0.484           0.501           0.592
oai_seed1                0.490           0.508           0.604
oai_seed2                0.484           0.502           0.595
mendeley                 0.377           0.408  

## Interpretation

If the four-category accuracies are substantially higher than the five-class figure, the difference attributable to grading scheme is quantified and can be reported directly. The remaining difference from a single-dataset result reflects the harder multi-dataset (LODO) setting, which is the core contribution of this work and is not expected to match a single-dataset number.